In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('spam.csv', encoding='latin-1')

In [3]:
# Keep only useful columns
df = df[['v1', 'v2']]

# Rename them
df.columns = ['label', 'text']

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df['label'].value_counts() ##Used to look if the data is balanced or not , the result shows that it's overwelhmingly Unbalanced

label
ham     4825
spam     747
Name: count, dtype: int64

In [5]:
#Encode the target column
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
#text preprocessing
df['text'] = df['text'].str.lower()

In [7]:
##TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['text'])

In [8]:
##Convert to arrays
X = X.toarray()
y = df['label'].values

In [9]:
#Train - Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
#convert to pytorch tensors
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [12]:
#Handle class imbalance by adjusting training weights using this equation
num_ham = (y_train == 0).sum().item()
num_spam = (y_train == 1).sum().item()

pos_weight = torch.tensor([num_ham / num_spam])

In [13]:
##This is our first trial , a 2 layers approach
import torch.nn as nn

class Model1(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [14]:
##This is close to a deep learning approach using 3 dense layers. we want to see if it gives better metrics here
class Model2(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [15]:
#Initialize model , loss , optimizer.
input_size = X_train.shape[1]

model = Model1(input_size)  # change to Model2 for second trial

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
#Training loop
epochs = 10

for epoch in range(epochs):
    model.train()
    
    logits = model(X_train).squeeze()
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 1.2011
Epoch 2, Loss: 1.1988
Epoch 3, Loss: 1.1961
Epoch 4, Loss: 1.1928
Epoch 5, Loss: 1.1888
Epoch 6, Loss: 1.1840
Epoch 7, Loss: 1.1785
Epoch 8, Loss: 1.1720
Epoch 9, Loss: 1.1647
Epoch 10, Loss: 1.1563


In [17]:
##model evaluation
model.eval()

with torch.no_grad():
    logits = model(X_test).squeeze()
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

accuracy = (preds == y_test).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.9793722033500671


In [18]:
#But accuracy is never a good metric when you have unbalanced data so it's not we are looking after here
#We should also look for precesion , recall , f1-score
from sklearn.metrics import classification_report

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

         0.0       0.98      1.00      0.99       965
         1.0       0.97      0.87      0.92       150

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [19]:
##now we test the other approach
model2 = Model2(input_size)

In [20]:
## new optimizer for the new model
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)

In [21]:
#Model 2 training
epochs = 10

for epoch in range(epochs):
    model2.train()
    
    logits = model2(X_train).squeeze()
    loss = criterion(logits, y_train)
    
    optimizer2.zero_grad()
    loss.backward()
    optimizer2.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 1.2007
Epoch 2, Loss: 1.1994
Epoch 3, Loss: 1.1976
Epoch 4, Loss: 1.1948
Epoch 5, Loss: 1.1910
Epoch 6, Loss: 1.1859
Epoch 7, Loss: 1.1794
Epoch 8, Loss: 1.1713
Epoch 9, Loss: 1.1617
Epoch 10, Loss: 1.1511


In [22]:
model2.eval()

with torch.no_grad():
    logits = model2(X_test).squeeze()
    probs = torch.sigmoid(logits)
    preds2 = (probs > 0.5).float()

accuracy2 = (preds2 == y_test).float().mean()

print("Model 2 Accuracy:", accuracy2.item())


Model 2 Accuracy: 0.9775784611701965


In [23]:
from sklearn.metrics import classification_report

print(classification_report(y_test, preds2))

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       965
         1.0       0.93      0.91      0.92       150

    accuracy                           0.98      1115
   macro avg       0.96      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [24]:
##When we used deeper learning we got better results for recall and f1 which are the most important here since our dataset is imbalanced
##So that encourged us to go on and user more layers and try if it will overfit or give better results
##even though it's excepcted that the model will overfit , we want to experement what happens next to be sure that model2 is our best case here
##so now we try a 512-> 256 -> 128 ->64 -> 1 approach and look for results 
class Model3(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [25]:
model3 = Model3(input_size)
optimizer3 = torch.optim.Adam(model3.parameters(), lr=0.001)

In [26]:
epochs = 10

for epoch in range(epochs):
    model3.train()
    
    logits = model3(X_train).squeeze()
    loss = criterion(logits, y_train)
    
    optimizer3.zero_grad()
    loss.backward()
    optimizer3.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 1.2023
Epoch 2, Loss: 1.2011
Epoch 3, Loss: 1.1996
Epoch 4, Loss: 1.1972
Epoch 5, Loss: 1.1932
Epoch 6, Loss: 1.1871
Epoch 7, Loss: 1.1788
Epoch 8, Loss: 1.1682
Epoch 9, Loss: 1.1546
Epoch 10, Loss: 1.1384


In [27]:
model3.eval()

with torch.no_grad():
    logits = model3(X_test).squeeze()
    probs = torch.sigmoid(logits)
    preds3 = (probs > 0.5).float()

accuracy3 = (preds3 == y_test).float().mean()

print("Model 3 Accuracy:", accuracy3.item())

Model 3 Accuracy: 0.9721972942352295


In [28]:
print(classification_report(y_test, preds3))
##as excpected the numbers went down in this model

              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98       965
         1.0       0.90      0.89      0.90       150

    accuracy                           0.97      1115
   macro avg       0.94      0.94      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [30]:
from sklearn.metrics import precision_score, recall_score, f1_score

def get_metrics(y_true, preds):
    precision = precision_score(y_true, preds)
    recall = recall_score(y_true, preds)
    f1 = f1_score(y_true, preds)
    
    return precision, recall, f1

In [31]:
p1, r1, f1_1 = get_metrics(y_test, preds)
p2, r2, f1_2 = get_metrics(y_test, preds2)
p3, r3, f1_3 = get_metrics(y_test, preds3)

In [32]:
print("=== Model Comparison (Spam Class Metrics) ===\n")

print(f"Model 1 → Precision: {p1:.3f}, Recall: {r1:.3f}, F1: {f1_1:.3f}")
print(f"Model 2 → Precision: {p2:.3f}, Recall: {r2:.3f}, F1: {f1_2:.3f}")
print(f"Model 3 → Precision: {p3:.3f}, Recall: {r3:.3f}, F1: {f1_3:.3f}")

=== Model Comparison (Spam Class Metrics) ===

Model 1 → Precision: 0.970, Recall: 0.873, F1: 0.919
Model 2 → Precision: 0.925, Recall: 0.907, F1: 0.916
Model 3 → Precision: 0.899, Recall: 0.893, F1: 0.896
